# EMT IEEE-39 Bus Multi-Area Diakoptics Runtime Measurement

Loads four copies of the IEEE-39 bus system, merges them into one system and couples them in a ring. The `EMT_IEEE_39bus_Multi_CIM` example runs four variants and reports total time and the per-step solve times:

- monolithic (single MNA system matrix)
- monolithic + OpenMP level scheduler
- diakoptics (ring couplers as tear components -> four subsystems)
- diakoptics + OpenMP level scheduler

The coupling node voltages of the monolithic and diakoptics runs are compared to confirm both solvers agree and per-step solve times are analysed to compare the system-matrix recomputation triggered by the fault switch at t = 0.005 s.

## Background: Why diakoptics, and what changed with KLU

**Original Idea:** A large network is torn into subnetworks; the coupling is a small dense Schur complement. On a topology change (switch/fault) only the affected subnetwork block must be refactorized instead of the whole system matrix — a benefit even without parallelization (https://doi.org/10.1049/icp.2025.4299).

**DPsim:** DPsim's KLU adapter now already supports partial refactorization, which the monolithic KLU solver uses. This also lets the monolithic solver refactorize cheaply at a switch.

**Measurements:** The dense columns use `Eigen::PartialPivLU` (pre-KLU path), the KLU columns use KLU on both solvers:

| Stage | MNA before (dense) | Diakoptics before (dense) | MNA + KLU | Diakoptics + KLU |
|-------|----------------------|-----------------------------|-------------|--------------------|
| **Switch step [ms]** | 161.0 | 11.7 | **3.6** | 12.0 |
| **Avg step [ms]**    | 5.81  | 2.04 | 4.28 | **1.71** |

So while diakoptics improved the switch step by ~13.8x (for the given example), switching to KLU improved the switching step even more significantly (45× speedup). As the diakoptics approach speeds up non-switching step times compared to monolithic KLU, it's still interesting as for most simulations switching /recomputation happens rarely.


*(Numbers are multiple tests on one machine; absolute values drift with load, but ratios and effects are stable.
All other data is recomputed live from this run.)*

## Setup

- Domain: EMT, three-phase (ABC)
- Linear solver: KLU, system matrix recomputation enabled
- Grid: 4 copies of IEEE-39, coupled in a ring
- Subsystems (diakoptics): 4 (ring couplers are used as tear components)
- Timestep: 10 us, final time: 0.02 s -> 2000 steps
- Fault: ground switch at bus N05 of area 0 closes at t = 0.005 s

## Run

Prints the total time of each variant and writes per-step solve times to `logs/<variant>/<variant>_step_times.log`.
Run takes approx. 60s.

In [ ]:
import os
import subprocess
from pathlib import Path

# Locate the repository top and build directory (supports in-source and out-of-source builds)
top = Path(
    subprocess.run(
        ["git", "rev-parse", "--show-toplevel"],
        capture_output=True,
        text=True,
        check=True,
    ).stdout.strip()
)
build = next(
    b
    for b in (top / "build", top.parent / "build", top.parent.parent / "build")
    if (b / "_deps/cim-data-src/IEEE-39").is_dir()
)
env = {**os.environ, "CIMPATH": str(build / "_deps/cim-data-src/IEEE-39")}

# Locate the executable (supports multi-config subdirs and the .exe suffix)
exe = next(
    p
    for p in (build / "dpsim/examples/cxx").rglob("EMT_IEEE_39bus_Multi_CIM*")
    if p.is_file() and p.suffix in ("", ".exe")
)

result = subprocess.run([str(exe)], env=env, capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise SystemExit(f"{exe.name} exited with code {result.returncode}")

## Validate results

Overlay the coupling node voltages of the monolithic and diakoptics runs. They must match.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import villas.dataprocessing.readtools as rt
import villas.dataprocessing.plottools as pt

dt = 10e-6
fault_time = 0.005
fault_step = int(round(fault_time / dt))

variants = {
    "monolithic": "IEEE-39bus_multi",
    "monolithic + OpenMP": "IEEE-39bus_multi_openMP",
    "diakoptics": "IEEE-39bus_multi_diakoptics",
    "diakoptics + OpenMP": "IEEE-39bus_multi_diakoptics_openMP",
}

In [ ]:
mono = rt.read_timeseries_dpsim("logs/IEEE-39bus_multi/IEEE-39bus_multi.csv")
diakoptics = rt.read_timeseries_dpsim(
    "logs/IEEE-39bus_multi_diakoptics/IEEE-39bus_multi_diakoptics.csv"
)

for i in range(0, 4):
    varname = "area" + str(i) + ".V_0"
    pt.set_timeseries_labels(mono[varname], varname + " monolithic")
    pt.set_timeseries_labels(diakoptics[varname], varname + " diakoptics")
    pt.plot_timeseries(i + 1, mono[varname])
    pt.plot_timeseries(i + 1, diakoptics[varname], "--")
    plt.title("IEEE-39 with 4 areas coupled in a ring")
    plt.xlabel("Time [s]")
    plt.ylabel("Voltage [V]")
    diff = max(abs(mono[varname].values - diakoptics[varname].values))
    print(f"area{i}.V_0 max|mono - diakoptics| = {diff:.3e} V")

plt.show()

## Critical time: per-step solve times

The diakoptics solver replaces one large system matrix by several smaller subsystem matrices, so each timestep is cheaper to solve. The cost moves to the switch event at t = 0.005 s, where the (Schur-complement) system matrix has to be recomputed. The table and plot below quantify both effects:

- `solve time [s]`: sum of all step times (excludes initialisation, includes the switch)
- `avg step [ms]`: average time per timestep
- `switch step [ms]`: cost of the timestep that handles the fault -> matrix recomputation
- `post-fault avg [ms]`: average step time after the fault

In [ ]:
step_times = {}
rows = []
for label, sim in variants.items():
    v = np.loadtxt(f"logs/{sim}/{sim}_step_times.log", skiprows=1)
    step_times[label] = v
    switch_window = v[fault_step - 2 : fault_step + 3]
    rows.append(
        {
            "variant": label,
            "solve time [s]": v.sum(),
            "avg step [ms]": v.mean() * 1e3,
            "switch step [ms]": switch_window.max() * 1e3,
            "post-fault avg [ms]": v[fault_step + 5 :].mean() * 1e3,
        }
    )

table = pd.DataFrame(rows).set_index("variant")
table["solve speedup"] = (
    table.loc["monolithic", "solve time [s]"] / table["solve time [s]"]
)
table.round(3)

In [ ]:
t = np.arange(len(next(iter(step_times.values())))) * dt

plt.figure(figsize=(9, 5))
for label, v in step_times.items():
    plt.plot(t, v * 1e3, label=label, linewidth=0.7)
plt.axvline(fault_time, color="k", linestyle=":", linewidth=1, label="switch event")
plt.yscale("log")
plt.xlabel("Time [s]")
plt.ylabel("Step time [ms]")
plt.title("Per-step solve time - spike at the switch is the matrix recomputation")
plt.legend()
plt.show()